# MiniProject3 Part 1: Independent Review

Builds `dlong37_validation_log.csv` for the ten projects in dpate172's MiniProject2 (`dpate127_MiniProject2/`), following Part 1 of `README.md` and the layout of `netid_validation_log_example.csv`.

Placeholders used in the CSV:
- `N/A` : a field that is not filled in until Part 2 (`AdjudicatedValue`, `ResolutionType`, `PostDiscussAgreementStatus`, `Comments`)

`WOCProjectID`, `ProjectOwnerNetID`, `ValidatorNetID`, `Field` and `AuthorValue` are extracted from the MiniProject2 files. No validator judgements are written here.

In [66]:
import csv
import re
from pathlib import Path

BASE = Path('.')
MP2_DIR = BASE / 'dpate172_MiniProject2'

VALIDATOR_NETID = 'dlong37'
OWNER_NETID = 'dpate172'

OUT_FILE = BASE / f'{VALIDATOR_NETID}_validation_log.csv'
DELIM = ';'

NOT_YET = 'N/A'  # placeholder: not needed until Part 2

## 1. Load the owner's projects

- `<owner>_project_stats.csv` has the author's value for every field (one row per project).
- `net2prj.csv` maps each WoC project name to its GitHub URL; the example log uses the GitHub URL in `WOCProjectID`.

In [67]:
with open(MP2_DIR / f'{OWNER_NETID}_project_stats.csv', newline='') as f:
    stats = list(csv.DictReader(f, delimiter=';'))

with open(MP2_DIR / 'net2prj.csv', newline='') as f:
    woc_to_url = {r['WoC']: r['GH'] for r in csv.DictReader(f) if r['netID'] == OWNER_NETID}

projects = [r['Project'] for r in stats]
missing = [p for p in projects if p not in woc_to_url]
assert len(projects) == 10 and not missing, (len(projects), missing)

for p in projects:
    print(f'{p:28s} {woc_to_url[p]}')

ketch_nodepy                 https://github.com/ketch/nodepy
mouseland_suite2p            https://github.com/MouseLand/suite2p
esri_geoportal-server        https://github.com/Esri/geoportal-server
hpparvi_pytransit            https://github.com/hpparvi/PyTransit
sundmanbo_opencalphad        https://github.com/sundmanbo/opencalphad
stack-of-tasks_pinocchio     https://github.com/stack-of-tasks/pinocchio
microsoft_fhir-server        https://github.com/microsoft/fhir-server
rapidsai_cucim               https://github.com/rapidsai/cucim
agnwinds_python              https://github.com/agnwinds/python
csgillespie_powerlaw         https://github.com/csgillespie/poweRlaw


## 2. Build the rows

Per README step 3, all 10 projects are listed for one field before moving to the next field. Column order follows `netid_validation_log_example.csv`.

In [68]:
COLUMNS = ['WOCProjectID', 'ProjectOwnerNetID', 'ValidatorNetID', 'Field',
           'AuthorValue', 'ValidatorValue', 'PreDiscussAgreementstatus(Yes/No)',
           'KeyEvidence', 'AdjudicatedValue', 'ResolutionType',
           'PostDiscussAgreementStatus', 'Comments']

FIELDS = [('HasRecovered',    'HasRecovered'),
          ('WhyRecovered',    'WhyRecovered'),
          ('ActivityPattern', 'ActivityPattern'),
          ('CurrentStatus',   'Currentstatus'),
          ('RecentThemes',    'RecentThemes')]

HAS_RECOVERED = {
    "ketch_nodepy": "Yes",
    "mouseland_suite2p": "N/A active project",
    "esri_geoportal-server": "No",
    "hpparvi_pytransit": "Yes",
    "sundmanbo_opencalphad": "Yes",
    "stack-of-tasks_pinocchio": "N/A active project",
    "microsoft_fhir-server": "N/A active project",
    "rapidsai_cucim": "N/A active project",
    "agnwinds_python": "N/A active project",
    "csgillespie_powerlaw": "Yes"
}

WHY_RECOVERED = {
    "ketch_nodepy": "Maintainer David Ketcheson resumed in Sept 2024 with a code change improving the SSP(5,4) coefficients (1ea45f7) plus a README downloads-badge merge. This is light maintenance by the original maintainer after pre-gap activity that was mostly documentation cleanup by a contributor.",
    "mouseland_suite2p": "N/A active project",
    "esri_geoportal-server": "N/A",
    "hpparvi_pytransit": "Development resumed in Jan 2016 with a bug fix for the z==k case (ba60c30) and a follow-up cleanup. Later commits switched to a faster elliptic-integral routine and merged fork contributions, so recovery came from fixing a bug and then improving performance.",
    "sundmanbo_opencalphad": "Maintainer Bo Sundman restarted in June 2024 by implementing multicomponent MQMQA (bef99c4) and continued in Oct 2024 with preparation for a major MQMQA implementation, building on the new-model work (XML format, CVM) he had been preparing before the gap. A new contributor (zhongjingjogy) also began adding Linux build and GitHub Actions commits in Oct 2024.",
    "stack-of-tasks_pinocchio": "N/A active project",
    "microsoft_fhir-server": "N/A active project",
    "rapidsai_cucim": "N/A active project",
    "agnwinds_python": "N/A active project",
    "csgillespie_powerlaw": "Activity resumed in Jan 2024 with a vignette-engine fix (ad80bcd) and continued in Feb 2025 with a roxygen2 update, a pkgdown site, and improved lognormal docs. Recovery came from maintenance and documentation work on the existing package, after the last pre-gap commits in 2020 and a GitHub Actions test in 2021."
}

ACTIVITY_PATTERN = {
    "ketch_nodepy": "irregular",
    "mouseland_suite2p": "declining",
    "esri_geoportal-server": "declining",
    "hpparvi_pytransit": "cyclical",
    "sundmanbo_opencalphad": "declining",
    "stack-of-tasks_pinocchio": "irregular",
    "microsoft_fhir-server": "irregular",
    "rapidsai_cucim": "cyclical",
    "agnwinds_python": "cyclical",
    "csgillespie_powerlaw": "declining"
}

CURRENT_STATUS = {
    "ketch_nodepy": "Inactive",
    "mouseland_suite2p": "Active",
    "esri_geoportal-server": "Inactive",
    "hpparvi_pytransit": "Inactive",
    "sundmanbo_opencalphad": "Inactive",
    "stack-of-tasks_pinocchio": "Inactive",
    "microsoft_fhir-server": "Inactive",
    "rapidsai_cucim": "Active",
    "agnwinds_python": "Inactive",
    "csgillespie_powerlaw": "Inactive"
}

RECENT_THEMES = {
    "ketch_nodepy": "Documentation updates:Bug fixes",
    "mouseland_suite2p": "Feature development:Bug fixes",
    "esri_geoportal-server": "Documentation updates:Dependency updates",
    "hpparvi_pytransit": "Feature development:Bug fixes",
    "sundmanbo_opencalphad": "Other:Bug fixes",
    "stack-of-tasks_pinocchio": "Automated bot contributions:Feature development",
    "microsoft_fhir-server": "Dependency updates:Feature development",
    "rapidsai_cucim": "Automated bot contributions:Dependency updates",
    "agnwinds_python": "Bug fixes:Documentation updates",
    "csgillespie_powerlaw": "Bug fixes:Release"
}

PRE_DISCUSS_STATUS = {
    "HasRecovered": {
        "ketch_nodepy": "No",
        "mouseland_suite2p": "No",
        "esri_geoportal-server": "Yes",
        "hpparvi_pytransit": "Yes",
        "sundmanbo_opencalphad": "Yes",
        "stack-of-tasks_pinocchio": "No",
        "microsoft_fhir-server": "Yes",
        "rapidsai_cucim": "Yes",
        "agnwinds_python": "No",
        "csgillespie_powerlaw": "Yes"
    },
    "WhyRecovered": {
        "ketch_nodepy": "Yes",
        "mouseland_suite2p": "No",
        "esri_geoportal-server": "Yes",
        "hpparvi_pytransit": "Yes",
        "sundmanbo_opencalphad": "Yes",
        "stack-of-tasks_pinocchio": "No",
        "microsoft_fhir-server": "Yes",
        "rapidsai_cucim": "Yes",
        "agnwinds_python": "No",
        "csgillespie_powerlaw": "Yes"
    },
    "ActivityPattern": {
        "ketch_nodepy": "Yes",
        "mouseland_suite2p": "Yes",
        "esri_geoportal-server": "Yes",
        "hpparvi_pytransit": "No",
        "sundmanbo_opencalphad": "Yes",
        "stack-of-tasks_pinocchio": "Yes",
        "microsoft_fhir-server": "No",
        "rapidsai_cucim": "No",
        "agnwinds_python": "No",
        "csgillespie_powerlaw": "Yes"
    },
    "CurrentStatus": {
        "ketch_nodepy": "Yes",
        "mouseland_suite2p": "Yes",
        "esri_geoportal-server": "No",
        "hpparvi_pytransit": "No",
        "sundmanbo_opencalphad": "No",
        "stack-of-tasks_pinocchio": "No",
        "microsoft_fhir-server": "No",
        "rapidsai_cucim": "Yes",
        "agnwinds_python": "Yes",
        "csgillespie_powerlaw": "Yes"
    },
    "RecentThemes": {
        "ketch_nodepy": "Yes",
        "mouseland_suite2p": "Yes",
        "esri_geoportal-server": "Yes",
        "hpparvi_pytransit": "No",
        "sundmanbo_opencalphad": "Yes",
        "stack-of-tasks_pinocchio": "No",
        "microsoft_fhir-server": "No",
        "rapidsai_cucim": "Yes",
        "agnwinds_python": "Yes",
        "csgillespie_powerlaw": "Yes"
    }
}

KEY_EVIDENCE = {
    "HasRecovered": {
        "ketch_nodepy": "Plot shows ~0 commits/month from Oct 2023 to Aug 2024, then 2 commits in Sep 2024 (1ea45f7 coefficient fix and merge 0f071ac with a README badge update). The pre-gap README announces no shutdown, but only 2 commits follow, so the recovery is thin.",
        "mouseland_suite2p": "Plot shows commits in nearly every month from 2018-07 onward; the only dip is a ~1-month lull around Nov 2024 (0-2 commits) before climbing back to ~20-65/month in 2025. A 1-month dip is not a real gap, so I treated it as an active project.",
        "esri_geoportal-server": "Plot shows zero commits from Nov 2024 to Jun 2025, then only 2 commits in Jul 2025 (README and index.html edits), far below the earlier 50-127/month peaks. The post-gap README reiterates archive status, no planned updates and successor repos, so there is no real recovery.",
        "hpparvi_pytransit": "Plot shows zero commits from ~Jun to Dec 2015, then a spike of ~28 in Jan 2016 and continued activity afterward. The first post-gap commit ba60c30 fixes the z==k bug reported in issue #12, and ~967 dated commits follow the gap.",
        "sundmanbo_opencalphad": "Plot shows zero commits Jan-May 2024, 1 commit in Jun 2024 (bef99c4, MQMQA), and a spike of ~41 in Oct 2024. changes.txt records slow, part-time development in 2023 and renewed MQMQA work on 2024-06-18.",
        "stack-of-tasks_pinocchio": "Plot shows commits in every month from 2014-08 to 2025 (typically 20-400/month) with only isolated one-month dips to ~0 (e.g., Dec 2014, early 2017). There is no multi-month gap, so it is an active project.",
        "microsoft_fhir-server": "Plot shows ~70-450 commits in every month from 2018-08 to 2025-10 with no zero-commit month (the final point is a partial month). There is no gap to recover from.",
        "rapidsai_cucim": "Plot shows 7-126 commits in every month from 2021-04 on, never zero; the lowest point is ~7 in Nov 2024. There is no gap to recover from.",
        "agnwinds_python": "Plot shows 30-130 commits/month almost throughout 2013-2024, with only a 2-month dip (Apr-May 2023) and a brief low in Aug 2016. Activity rebounds to ~119 commits by late summer 2023, so I treated it as an active project.",
        "csgillespie_powerlaw": "Plot shows essentially zero commits from ~Jun 2020 to Jan 2025 (stray 1-2 commit months in Nov 2021 and Jan 2024), then a burst of ~17 in Feb 2025 (pkgdown, roxygen2 update, PR #103). The recovery is real but thin: only 18 dated commits follow the gap."
    },
    "WhyRecovered": {
        "ketch_nodepy": "Commit 1ea45f7 (Sep 2024, 'Improved coefficients for SSP(5,4)') raises coefficient precision, followed by merge 0f071ac of a README downloads-badge update. The pre-gap README (147d082) invites contributions and announces no shutdown.",
        "mouseland_suite2p": "No recovery trigger to explain: the plot shows steady monthly commits with only a ~1-month dip around Nov 2024.",
        "esri_geoportal-server": "Only 2 post-gap commits (cd9bdd2f 'Update README.md', 3014a9ce 'Update index.html'). They restate archive status, no planned updates and successor repos, and change website text and badges, with no application changes. No recovery trigger.",
        "hpparvi_pytransit": "Issue #12 (opened 2015-11-17, closed 2016-01-13) reports the z==k bug, and ba60c30 ('A simple fix for the bug occurring when z==k') is the first post-gap commit, followed by cleanup 3ae8da7 and a faster elliptic-integral routine (fd6b3f8). The pre-gap README announces no hiatus, so the cause of the pause itself is unknown.",
        "sundmanbo_opencalphad": "bef99c4 (2024-06-18, 'Start implementing multicomponent MQMQA') and a changes.txt entry mark renewed MQMQA work after slow, part-time development in 2023, and fc124e7 (Oct 2024) prepares a major MQMQA implementation. Issue #57 (Oct 2024 comment) has Sundman citing other tasks, and contributor zhongjingjogy adds Linux build and GitHub Actions commits from 2024-10-09.",
        "stack-of-tasks_pinocchio": "No recovery to explain: the plot shows commits in every month, with only isolated one-month dips (e.g., Dec 2014).",
        "microsoft_fhir-server": "No recovery to explain: the plot shows 70-450 commits in every month with no zero-commit month.",
        "rapidsai_cucim": "No recovery to explain: the plot shows 7-126 commits in every month with no zero-commit month.",
        "agnwinds_python": "No real recovery to explain: only a 2-month dip (Apr-May 2023) in otherwise continuous activity, with commits back to ~119/month by late summer 2023.",
        "csgillespie_powerlaw": "The first post-gap commit ad80bcd ('fix: Package engine in vignette', Jan 2024) was merged as PR #103 on 2025-02-02, followed by a roxygen2 7.2.3 to 7.3.2 regeneration (5dd00c7) and a pkgdown site deployed to gh-pages via GitHub Actions. The post-gap README describes CRAN install and vignettes with no explanation of the hiatus."
    },
    "ActivityPattern": {
        "ketch_nodepy": "Plot shows isolated bursts (~51 in Oct 2012, ~56-57 in Mar-Apr 2015, ~57 in mid-2020) separated by long stretches of 0-6 commits/month, with no sustained trend or fixed period.",
        "mouseland_suite2p": "Largest peaks come early (~380 in Aug 2018, ~310 in Jun-Jul 2020); from 2021 on, monthly counts settle to roughly 5-50, with a modest uptick in 2025-26.",
        "esri_geoportal-server": "Peaks of ~127 (mid-2013) and 60-90 (2014-2016) fall to under ~25/month after 2017 and to near zero by 2023-2025, with only 2 commits in Jul 2025.",
        "hpparvi_pytransit": "Plot shows recurring waves: bursts (~36 in early 2015, ~85 in Mar 2020, ~41 in Feb 2024, ~32 in early 2026) alternating with quiet stretches (2017-18, 2022-23, 2025) rather than one overall trend.",
        "sundmanbo_opencalphad": "Steady 5-19 commits/month in 2015-2017 decline to 0-3 by 2022-2023. The Oct 2024 spike (~41) is a one-off after zero commits in Jan-May 2024.",
        "stack-of-tasks_pinocchio": "Monthly counts swing between ~0 and ~413 (Aug 2018), with other surges in early 2020 (~344) and 2021, and no steady trend or fixed period.",
        "microsoft_fhir-server": "Counts oscillate between ~70 and ~450 commits/month (peak in spring 2021) with no sustained rise, fall or fixed cycle.",
        "rapidsai_cucim": "Counts oscillate between ~7 and ~126/month with repeated peaks (~106 in late 2021, ~126 in Nov 2022, ~112 in Oct 2023, ~103 in Jan 2024) and troughs between them.",
        "agnwinds_python": "Plot shows repeated rises and dips over 2013-2024: peaks of ~104 (2015), ~119 (early 2018), ~121 (mid-2019), ~134 (spring 2020) and ~119 (2023), with troughs near 0-2 in Aug 2016 and Apr-May 2023.",
        "csgillespie_powerlaw": "Peaks of ~50 in spring 2013 fall to under ~25/month through 2014-2020, then essentially zero from mid-2020 to Jan 2025 apart from a burst of ~17 in Feb 2025."
    },
    "CurrentStatus": {
        "ketch_nodepy": "Newest commit is 2024-09-04 (merge 0f071ac in recent_commits.csv), about 19 months before the 2026-09-30 reference date and well before the 2026-04-01 cutoff.",
        "mouseland_suite2p": "Plot and project_summary.csv show commits through 2026-04-18, after the 2026-04-01 cutoff. recent_commits.csv (capped at 2025-09-30) also shows 10 commits from 2025-07-02 to 2025-08-30.",
        "esri_geoportal-server": "Newest commit is 2025-07-13 ('Update index.html', 3014a9ce), after a 2025-07-08 README edit, both well before the 2026-04-01 cutoff. Earlier, 2024-11 to 2025-06 had zero commits, and the README says archived with no planned updates.",
        "hpparvi_pytransit": "CHECK: newest commit in project_summary.csv is 2026-03-18, two weeks before the 2026-04-01 cutoff, so the date rule gives Inactive; the plot ends 2026-03 with ~30 commits, and GitHub's lastGHCommitDate in project_stats.csv is 2026-08-18, which would support Active.",
        "sundmanbo_opencalphad": "Newest commit is 2025-05-15 (plot ends 2025-05; recent_commits.csv shows 'Fixed a problem with example' on 2025-05-13), about 10 months before the 2026-04-01 cutoff.",
        "stack-of-tasks_pinocchio": "Newest commit is 2025-11-05 (plot ends 2025-11, with only ~17 commits in the final partial month), before the 2026-04-01 cutoff.",
        "microsoft_fhir-server": "Newest commit is 2025-11-01 (plot ends 2025-11, with the last point near 0), before the 2026-04-01 cutoff.",
        "rapidsai_cucim": "Plot and project_summary.csv show commits through 2026-06-11 (plot x-axis ends 2026-06, with ~11 commits in the final month), after the 2026-04-01 cutoff.",
        "agnwinds_python": "Newest commit is 2024-10-22 ('Provide a more complete README file'), almost 2 years before the 2026-09-30 reference date.",
        "csgillespie_powerlaw": "Newest commit is 2025-02-03 ('CRAN release (#104)'), before the 2026-04-01 cutoff. The next most recent is also Feb 2025, then May 2020."
    },
    "RecentThemes": {
        "ketch_nodepy": "8 of 10 commits are README or Sphinx docs fixes. The only code commit is 'Improved coefficients for SSP(5,4)'.",
        "mouseland_suite2p": "Features include Cellpose 4/Cellpose-SAM support and GPU use. Fixes include Python 3.11 fix, the #1179 time-step cast, and default settings.",
        "esri_geoportal-server": "5 README or index.html edits. 4 commits are org.json or commons-fileupload bumps, including their merges.",
        "hpparvi_pytransit": "Features include slope_order, TSModel accuracy, an example notebook, and pyrr made optional. Fixes include NumPy 2.0 and SciPy compatibility and a dtype fix.",
        "sundmanbo_opencalphad": "5 of 10 commits are housekeeping: new Mac and Linux install tests, deleting redundant files. 3 commits are fixes. Only 2 commits are real features (MQMQA/PLOT and XTDB).",
        "stack-of-tasks_pinocchio": "Dependabot, github-actions[bot] and their merges account for 6 commits. The viser, nix and example work account for 4.",
        "microsoft_fhir-server": "Dependency updates include action bumps, a shared-package bump to 10.0.68, and the AzureAD to MS Graph migration. Features include $docref, transaction cencellation, and the search-parameter cache service.",
        "rapidsai_cucim": "4 gpuCI forward-merges plus an automated DOC v25.12 Updates [skip ci]. Dependency work includes the libcufile relaxation, adding pip, and the RMM and pre-commit cleanup.",
        "agnwinds_python": "Fixes include the istat logic for bug #1093 and the xband fixes. Docs include three README or docs commits.",
        "csgillespie_powerlaw": "Fixes include the vignette engine, the lognormal tail issue (#88), and test partial-arg matching. Releases include the CRAN release and the News/Description bumps."
    }
}

# my ValidatorValue dictionaries, keyed by the name written to the log's Field column
VALIDATOR_VALUES = {
    'HasRecovered': HAS_RECOVERED,
    'WhyRecovered': WHY_RECOVERED,
    'ActivityPattern': ACTIVITY_PATTERN,
    'CurrentStatus': CURRENT_STATUS,
    'RecentThemes': RECENT_THEMES,
}
for name, d in [('VALIDATOR_VALUES', VALIDATOR_VALUES),
                ('PRE_DISCUSS_STATUS', PRE_DISCUSS_STATUS),
                ('KEY_EVIDENCE', KEY_EVIDENCE)]:
    assert set(d) == {field for field, _ in FIELDS}, name
    for field, values in d.items():
        assert set(values) == set(projects), (name, field, set(projects) ^ set(values))

rows = []
for field, stats_col in FIELDS:
    for r in stats:
        rows.append({
            'WOCProjectID': woc_to_url[r['Project']],
            'ProjectOwnerNetID': OWNER_NETID,
            'ValidatorNetID': VALIDATOR_NETID,
            'Field': field,
            'AuthorValue': r[stats_col].strip(),
            # Part 1 fields
            'ValidatorValue': VALIDATOR_VALUES[field][r['Project']],
            'PreDiscussAgreementstatus(Yes/No)': PRE_DISCUSS_STATUS[field][r['Project']],
            'KeyEvidence': KEY_EVIDENCE[field][r['Project']],
            # Part 2 fields
            'AdjudicatedValue': NOT_YET,
            'ResolutionType': NOT_YET,
            'PostDiscussAgreementStatus': NOT_YET,
            'Comments': NOT_YET,
        })

assert len(rows) == 50
len(rows)

50

## 3. Write `dlong37_validation_log.csv`

In [69]:
with open(OUT_FILE, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS, delimiter=DELIM)
    writer.writeheader()
    writer.writerows(rows)

print(f'Wrote {len(rows)} rows to {OUT_FILE}')

Wrote 50 rows to dlong37_validation_log.csv


## 4. Check the output

In [70]:
with open(OUT_FILE, newline='') as f:
    check = list(csv.DictReader(f, delimiter=DELIM))

assert len(check) == 50 and list(check[0]) == COLUMNS

# each field block holds all 10 projects, in field order
for i, (field, _) in enumerate(FIELDS):
    block = check[i * 10:(i + 1) * 10]
    assert {r['Field'] for r in block} == {field}
    assert [r['WOCProjectID'] for r in block] == [woc_to_url[p] for p in projects]

for r in check[::10]:
    print(r['Field'], '->', r['WOCProjectID'], '|', r['AuthorValue'][:50])
print()
print(open(OUT_FILE).read().splitlines()[0])

HasRecovered -> https://github.com/ketch/nodepy | Partly: two commits on September 4, 2024; sustaine
WhyRecovered -> https://github.com/ketch/nodepy | The visible trigger was a correction to SSP(5,4) c
ActivityPattern -> https://github.com/ketch/nodepy | irregular
CurrentStatus -> https://github.com/ketch/nodepy | Inactive
RecentThemes -> https://github.com/ketch/nodepy | Documentation updates:Bug fixes

WOCProjectID;ProjectOwnerNetID;ValidatorNetID;Field;AuthorValue;ValidatorValue;PreDiscussAgreementstatus(Yes/No);KeyEvidence;AdjudicatedValue;ResolutionType;PostDiscussAgreementStatus;Comments
